# 安装 verl + vllm-ascend

本节将安装 RL 训练所需的三个核心组件：vLLM（推理引擎）、vLLM-Ascend（NPU 适配层）和 verl（RL 训练框架）。

安装完成后，你还将应用 Wordle AgentLoop 补丁，为后续训练做好准备。

---

## 1. 组件关系

Wordle RL 训练需要三个核心组件协同工作：

In [ ]:
import os

# CANNLab 进入后默认工作目录为 cann-recipes-train 仓库根目录
# 训练文件位于 llm_rl/qwen3_wordle/，课程 notebook 位于同级 cann-learning-hub 仓库
print(f'当前工作目录: {os.getcwd()}')
assert os.path.isdir('llm_rl/qwen3_wordle'), '请确认在 cann-recipes-train 根目录下启动 Jupyter'
print('目录检查通过: llm_rl/qwen3_wordle/ 存在')


<img src="./images/stack_layers.png" alt="Wordle RL 环境栈" width="90%">

这三个组件是分层关系：外层的 verl 负责调度 RL 训练流程，中层的 vLLM / vllm-ascend 负责在 rollout 阶段生成模型回复，底层的 Ascend NPU 和 CANN 提供实际计算能力。

**verl** 负责管理训练循环：收集 rollout → 计算 reward → 计算 advantage → 更新策略。

**vLLM** 负责生成 rollout：在训练过程中让模型生成文本（如 Wordle 猜词），以 OpenAI API 兼容接口提供服务。

**vLLM-Ascend** 是 vLLM 的 NPU 适配层，让 vLLM 能在 Ascend NPU 上运行。

---

## 2. 版本配套表

| 组件 | 版本 | 说明 |
|------|------|------|
| CANN | 9.0.0 | NPU 驱动 + 工具链 |
| torch | 2.9.0 | PyTorch |
| torch_npu | 2.9.0 | PyTorch NPU 适配 |
| vllm | 0.18.0 | 推理引擎 |
| vllm-ascend | 0.18.0 | vLLM NPU 适配 |
| verl | 0.8.0 | RL 训练框架 |

---

## 3. 安装步骤

CANNLab 进入后默认工作目录为 `cann-recipes-train` 仓库根目录，目录结构如下：

```text
<home>/
├── cann-recipes-train/        <- CANNLab 默认工作目录（本仓库）
│   └── llm_rl/qwen3_wordle/   <- 训练工作目录
└── cann-learning-hub/         <- 课程 notebook（与 cann-recipes-train 同级）
```

下方命令中：vLLM / vLLM-Ascend 安装到 `/home/developer/`（避免与训练目录产生 Python 导入冲突）；verl 和训练依赖安装到 `llm_rl/qwen3_wordle/` 下。

### 3.1 安装 vLLM

In [ ]:
%%bash
# vLLM 安装到 /home/developer/ 下，避免与训练工作目录产生 Python 导入冲突
cd /home/developer
git clone --depth 1 --branch v0.18.0 https://github.com/vllm-project/vllm
cd vllm && VLLM_TARGET_DEVICE=empty pip install -v -e .


### 3.2 安装 vLLM-Ascend

In [ ]:
%%bash
cd /home/developer
git clone --depth 1 --branch v0.18.0 https://github.com/vllm-project/vllm-ascend.git
cd vllm-ascend && git submodule update --init --recursive && pip install -v -e .


### 3.3 安装 verl

In [ ]:
%%bash
cd llm_rl/qwen3_wordle
git clone --depth 1 --branch v0.8.0 https://github.com/verl-project/verl
cd verl && pip install -v -e .


### 3.4 安装 Wordle 训练依赖

verl 会自动安装大部分依赖（pyarrow、pandas、ray、wandb 等），但 Wordle 训练还需要 nltk 和 textarena：

In [ ]:
%%bash
cd llm_rl/qwen3_wordle
pip install -r requirements.txt

# NLTK 数据（TextArena Wordle 依赖 pos_tag 过滤词性）
python3 -c "import nltk; nltk.download('words'); nltk.download('averaged_perceptron_tagger_eng')"


### 3.5 应用 Wordle AgentLoop 补丁

Wordle 训练需要一个自定义的 `WordleAgentLoop` 组件。通过 patch 将其注册到 verl 源码中：

In [ ]:
%%bash
cd llm_rl/qwen3_wordle/verl
git apply ../patches/0001-wordle-agent-loop.patch


---

## 4. 验证安装

运行以下命令，确认所有组件导入正常：

In [ ]:
%%bash
cd llm_rl/qwen3_wordle
python3 -c "
from verl.experimental.agent_loop.wordle_agent_loop import WordleAgentLoop
from importlib.metadata import version
print('verl:', version('verl'))
print('vllm:', version('vllm'))
print('All imports OK')
"


---

## 5. 常见问题

**Q: `No module named 'verl'`？**
A: 确认在 verl 目录下执行了 `pip install -v -e .`。

**Q: AgentLoop 补丁失败？**
A: 确认 verl 在 v0.8.0 版本（`git describe --tags`），补丁是针对此版本生成的。

**Q: NLTK 下载失败（SSL 错误）？**
A: 网络不通时需配置代理后重试。

---

## 课后练习

1. （判断题）vLLM 在 RL 训练中的主要作用是生成 rollout（模型回复），供奖励函数打分。

2. （判断题）verl 的 `pip install -e .` 会自动安装 pyarrow、pandas、ray 等依赖，无需手动安装。

3. （判断题）Wordle AgentLoop 补丁只需要复制 `wordle_agent_loop.py` 文件即可，不需要修改 verl 的 `__init__.py`。

4. （单选题）以下哪个组件负责在 RL 训练中管理推理、奖励、训练的循环？
    A. vLLM
    B. vLLM-Ascend
    C. verl
    D. CANN

5. （单选题）安装 verl 后还需要单独安装以下哪个依赖？
    A. pyarrow
    B. ray
    C. nltk
    D. pandas

In [ ]:
!cat ../cann-learning-hub/tutorials/rl_training_pipeline/01_environment_setup/answer/01.02_answer.txt